In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

# # Recurrent neural network model for fitting neural responses
class RNN(torch.nn.Module):
    def __init__(self, n, input_dim,beta,bias):
        super(RNN, self).__init__()

        self.n = n
        self.input_dim = 2
        self.beta = beta
        self.bias = bias
        self.activation = torch.nn.Softplus(beta =self.beta)
        # Hidden dynamics
        self.recurrent_layer = nn.Linear(self.n, self.n, bias=True)
        self.recurrent_layer.bias.data.normal_(mean=self.bias, std=0)
        self.recurrent_layer.bias.requires_grad = True

        self.input_layer = nn.Linear(self.input_dim, self.n, bias=False)
        self.output_layer = nn.Linear(self.n, self.output_dim, bias=False)



    # Dynamics

    def forward(self, u, dt, steps):
        batch_size = u.shape[0]
        states = torch.zeros(batch_size, 1, self.n, device=device)
        for i in range(steps - 1):
            state_new = (1 - dt) * states[:, i, :] + dt * (
                self.activation(self.recurrent_layer(states[:, i, :]) + self.input_layer(u)))
            states = torch.cat((states, state_new.unsqueeze_(1)), 1)

        return states

